## 1. Setup e Geração de Dados Sintéticos

In [9]:
# Imports básicos
import pandas as pd
import numpy as np
import kennard_stone as ks
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import plotly.express as px

# Configuração do plotly como backend
pd.options.plotting.backend = 'plotly'

In [100]:
# Geração de dados sintéticos
from synthetic import generate_synthetic_spectral_data

config = [
    {
        'nome': 'A',
        'n_amostras': 156,
        'picos': [250, 380, 550, 700, 850],
        'amp_media': 1.0,
        'amp_std': 0.3,
        'larg_media': 15.0,
        'larg_std': 2.0,
        'ruido_std': 0.04
    },
    {
        'nome': 'B',
        'n_amostras': 146,
        'picos': [50, 250, 380, 550, 850],
        'amp_media': 1.4,
        'amp_std': 0.5,
        'larg_media': 15.0,
        'larg_std': 1.8,
        'ruido_std': 0.035
    }
]

data_complete = generate_synthetic_spectral_data(
    configuracao_classes=config,
    n_pontos=500,
    x_min=1,
    x_max=1000,
    seed=0
)

print(f"Dataset shape: {data_complete.shape}")

Dataset shape: (302, 501)


In [101]:
# Separação em calibração e predição usando Kennard-Stone
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

XA_cal, XA_pred = ks.train_test_split(data_A.iloc[:, 1:], test_size=0.30)
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.iloc[:, 1:], test_size=0.30)
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-27 09:48:33,967 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-27 09:48:33,988 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-27 09:48:34,016 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

In [102]:
# Pré-processamento: Mean Centering
import preprocessings as prepr

Xcalclass_prep, mean_calclass = prepr.mc(Xcalclass)
Xpredclass_prep = Xpredclass - mean_calclass

In [103]:
# Construção do modelo PLS-DA
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass, 
    ycalclass,
    LVmax=1,
    Xpred=Xpredclass,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Predições contínuas para o conjunto de calibração
y_pred_cont = plsda_results[5].iloc[:, -1]
print(f"Predições contínuas shape: {y_pred_cont.shape}")

Predições contínuas shape: (211,)


In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv('XRF_databases/bank_notes/plsda/bank_notes.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1':'26.07']

In [2]:
# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'26.07'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'26.07'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-27 10:10:16,103 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-27 10:10:16,285 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [3]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv('XRF_databases/soil/plsda/soil.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1':'15']

In [2]:
# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'15'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'15'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

# Xcalclass_prep = Xcalclass
# Xpredclass_prep = Xpredclass

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-27 10:10:37,517 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-27 10:10:37,531 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-27 10:10:37,559 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [3]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

## 2. Definição das Zonas Espectrais

In [4]:
# # Definição das zonas espectrais (features e backgrounds)
# spectral_cuts = [
#     ('F1', 1.0, 100.0),
#     ('background1', 100.0, 200.0),
#     ('F2', 200.0, 300.0),
#     ('background2', 300.0, 330.0),
#     ('F3', 330.0, 430.0),
#     ('background3', 430.0, 500.0),
#     ('F4', 500.0, 600.0),
#     ('background4', 600.0, 660.0),
#     ('F5', 660.0, 750.0),
#     ('background5', 750.0, 815.0),
#     ('F6', 815.0, 890.0),
#     ('background6', 890.0, 1000.0)
# ]

# spectral_cuts = [
# ('background1', 1.0, 2.74),
# ('Ar ka + Ag L', 2.76, 3.47),
# ('Ca ka', 3.5, 3.91),
# ('Ca kb', 3.93, 4.24),
# ('Ti ka', 4.26, 4.72),
# ('Ti kb', 4.75, 5.13),
# ('background2', 5.16, 6.12),
# ('Fe ka', 6.15, 6.76),
# ('Fe kb', 6.79, 7.32),
# ('background3', 7.35, 7.78),
# ('Cu ka', 7.81, 8.29),
# ('Zn ka', 8.29, 8.80),
# ('Cu kb', 8.80, 9.26),
# ('Zn kb', 9.26, 10.00),
# ('background4', 10.00, 21.46),
# ('Ag ka scattering', 21.49, 22.71),
# ('background5', 22.74, 24.52),
# ('background6', 24.55, 26.07),
# ]

spectral_cuts = [
('background1', 1.0, 1.33),
('Al', 1.33, 1.63),
('Si', 1.63, 1.86),
('P', 1.86, 2.10),
('background2', 2.10, 2.19),
('S', 2.19, 2.44),
('background3', 2.44, 2.55),
('Rh L + Ar', 2.55, 3.10),
('background4', 3.10, 3.21),
('K', 3.21, 3.42),
('background5', 3.42, 3.53),
('Ca ka', 3.53, 3.84),
('Ca kb', 3.84, 4.14),
('background6', 4.14, 4.37),
('Ti ka', 4.37, 4.66),
('background7', 4.66, 4.75),
('Ti kb', 4.75, 5.12),
('Cr', 5.12, 5.77),
('Mn', 5.77, 6.02),
('background8', 6.02, 6.13),
('Fe ka', 6.13, 6.68),
('background9', 6.68, 6.80),
('Fe kb', 6.80, 7.30),
('background10', 7.30, 7.91),
('Cu', 7.91, 8.20),
('background11', 8.20, 10.69),
('Fe ka + Ti ka', 10.69, 11.14),
('background12', 11.14, 12.55),
('sum Fe' , 12.55, 13.1),
('background13', 13.1, 15.0)
] # soil

## 3. Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [5]:
def aggregate_spectral_zones_pca(spectral_zones_dict):
    """
    Agrega zonas espectrais usando PCA com 1 componente principal.
    
    Para cada zona espectral, ajusta uma PCA com 1 componente e extrai:
    - Scores: projeção das amostras na direção de máxima variância
    - Loadings: pesos de cada variável na PC1
    - Média: vetor de médias da zona (para reconstrução)
    - Variância Explicada: fração da variância capturada pela PC1
    
    Parameters
    ----------
    spectral_zones_dict : dict
        Dicionário retornado por extract_spectral_zones.
        Chaves = nomes das zonas, Valores = DataFrames com dados espectrais.
    
    Returns
    -------
    scores_df : pd.DataFrame
        DataFrame com scores da PC1 para cada zona (amostras x zonas).
    pca_info_dict : dict
        Dicionário com informações da PCA para cada zona:
        - 'loadings': vetor de loadings da PC1
        - 'mean': vetor de médias da zona
        - 'variance_explained': fração de variância explicada
        - 'columns': nomes das colunas originais (para reconstrução)
    """
    scores_dict = {}  # armazena scores de cada zona
    pca_info_dict = {}  # armazena informações para reconstrução
    
    for zone_name, zone_df in spectral_zones_dict.items():
        # ========================================
        # PASSO 1: Preparação dos dados
        # ========================================
        X_zone = zone_df.values  # converter para numpy array
        
        # ========================================
        # PASSO 2: Ajuste da PCA com 1 componente
        # ========================================
        pca = PCA(n_components=1)
        scores = pca.fit_transform(X_zone)  # scores da PC1 (n_samples, 1)
        
        # ========================================
        # PASSO 3: Extração das informações
        # ========================================
        loadings = pca.components_[0]  # loadings da PC1 (d_m,)
        mean_vector = pca.mean_  # vetor de médias (d_m,)
        variance_explained = pca.explained_variance_ratio_[0]  # fração de variância
        
        # ========================================
        # PASSO 4: Armazenamento
        # ========================================
        scores_dict[zone_name] = scores.flatten()  # converter para 1D
        
        pca_info_dict[zone_name] = {
            'loadings': loadings,
            'mean': mean_vector,
            'variance_explained': variance_explained,
            'columns': zone_df.columns.tolist(),  # nomes das colunas originais
            'pca_model': pca  # modelo PCA completo (para uso futuro)
        }
        
        # Log informativo
        print(f"Zona '{zone_name}': VE = {variance_explained:.2%}, "
              f"dim = {len(loadings)} variáveis")
    
    # Criar DataFrame com todos os scores
    scores_df = pd.DataFrame(scores_dict)
    
    return scores_df, pca_info_dict

In [6]:
def reconstruct_threshold_to_spectrum(threshold_value, zone_name, pca_info_dict):
    """
    Reconstrói um threshold escalar (no espaço dos scores) para o espaço 
    espectral original, gerando um "espectro de threshold" multivariado.
    
    Fórmula matemática:
        τ = mean + threshold_value * loadings
    
    Parameters
    ----------
    threshold_value : float
        Valor do threshold no espaço dos scores da PC1.
    zone_name : str
        Nome da zona espectral.
    pca_info_dict : dict
        Dicionário com informações da PCA (retornado por aggregate_spectral_zones_pca).
    
    Returns
    -------
    threshold_spectrum : pd.Series
        Espectro de threshold com índice = energias/comprimentos de onda originais.
    """
    # ========================================
    # Recuperar informações da PCA
    # ========================================
    pca_info = pca_info_dict[zone_name]
    loadings = pca_info['loadings']
    mean_vector = pca_info['mean']
    columns = pca_info['columns']
    
    # ========================================
    # Reconstrução: τ = mean + q * loadings
    # ========================================
    threshold_spectrum = mean_vector + threshold_value * loadings
    
    # Converter para Series com índice original
    threshold_spectrum = pd.Series(threshold_spectrum, index=columns, name=f'threshold_{threshold_value:.2f}')
    
    return threshold_spectrum

In [7]:
def plot_zone_with_threshold(zone_df, threshold_spectrum, zone_name, 
                              class_labels=None, title=None):
    """
    Plota os espectros de uma zona espectral junto com o espectro de threshold.
    
    Parameters
    ----------
    zone_df : pd.DataFrame
        DataFrame com dados espectrais da zona (amostras x variáveis).
    threshold_spectrum : pd.Series
        Espectro de threshold reconstruído.
    zone_name : str
        Nome da zona espectral.
    class_labels : pd.Series, optional
        Labels de classe para colorir as amostras.
    title : str, optional
        Título customizado para o gráfico.
    
    Returns
    -------
    fig : plotly.graph_objects.Figure
        Figura plotly interativa.
    """
    fig = go.Figure()
    
    # Converter índice para numérico (energias/comprimentos de onda)
    x_values = pd.to_numeric(zone_df.columns, errors='coerce')
    
    # ========================================
    # Plotar espectros das amostras
    # ========================================
    if class_labels is not None:
        # Colorir por classe
        colors = {'A': 'red', 'B': 'blue'}
        for idx, row in zone_df.iterrows():
            class_label = class_labels.iloc[idx] if idx < len(class_labels) else 'Unknown'
            fig.add_trace(go.Scatter(
                x=x_values,
                y=row.values,
                mode='lines',
                line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
                name=f'Class {class_label}',
                showlegend=False,
                hoverinfo='skip'
            ))
    else:
        # Sem coloração por classe
        for idx, row in zone_df.iterrows():
            fig.add_trace(go.Scatter(
                x=x_values,
                y=row.values,
                mode='lines',
                line=dict(color='blue', width=0.5),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # ========================================
    # Plotar espectro de threshold (destaque)
    # ========================================
    fig.add_trace(go.Scatter(
        x=x_values,
        y=threshold_spectrum.values,
        mode='lines',
        line=dict(color='gold', width=3, dash='dash'),
        name=f'Threshold Spectrum ({threshold_spectrum.name})'
    ))
    
    # ========================================
    # Layout do gráfico
    # ========================================
    fig.update_layout(
        title=title or f'Zona Espectral: {zone_name} com Threshold Multivariado',
        xaxis_title='Energia / Comprimento de Onda',
        yaxis_title='Intensidade',
        template='plotly_white',
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    
    return fig

## Pipeline SMeX com Agregação por PCA

In [10]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zona 'background1': VE = 11.28%, dim = 17 variáveis
Zona 'Al': VE = 79.62%, dim = 15 variáveis
Zona 'Si': VE = 92.02%, dim = 12 variáveis
Zona 'P': VE = 66.06%, dim = 13 variáveis
Zona 'background2': VE = 33.61%, dim = 5 variáveis
Zona 'S': VE = 43.37%, dim = 13 variáveis
Zona 'background3': VE = 24.20%, dim = 6 variáveis
Zona 'Rh L + Ar': VE = 45.17%, dim = 28 variáveis
Zona 'background4': VE = 29.12%, dim = 6 variáveis
Zona 'K': VE = 88.16%, dim = 11 variáveis
Zona 'background5': VE = 34.51%, dim = 6 variáveis
Zona 'Ca ka': VE = 99.00%, dim = 16 variáveis
Zona 'Ca kb': VE = 87.19%, dim = 16 variáveis
Zona 'background6': VE = 46.69%, dim = 12 variáveis
Zona 'Ti ka': VE = 89.96%, dim = 15 variáveis
Zona 'background7': VE = 79.76%, dim = 5 variáveis
Zona 'Ti kb': VE = 82.12%, dim = 19 variáveis
Zona 'Cr': VE = 54.08%, dim = 33 variáveis
Zona 'Mn': VE = 94.03%, dim = 13 variáveis
Zona 'background8': VE = 51.90%, dim = 6 variáveis
Zona 'Fe ka': VE = 55.81%, dim = 28 variáveis
Zona 'backgr

In [11]:
# A função predicates_by_quantiles funciona normalmente com os scores
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])

predicates_df = predicates_quantiles[0]  # DataFrame com regras dos predicados
predicate_indicator_df = predicates_quantiles[1]  # Matriz indicadora (amostras x predicados)
co_occurrence_matrix_df = predicates_quantiles[2]  # Matriz de co-ocorrência

# Bagging de predicados
seed = 0
training_samples = len(Xcalclass)
y_predicted_numeric = plsda_results[5].iloc[:, -1]  # predições numéricas do modelo

bags_result = exp.bagging_predicates(
    zone_sums_df=zone_scores_df,  # agora são scores da PCA
    y_predicted_numeric=y_predicted_numeric,
    predicates_df=predicates_df,
    n_bags=10,
    n_samples_per_bag=int(training_samples * 0.8),
    min_samples_per_predicate=int(training_samples * 0.2),
    replace=False,
    sample_bagging=True,
    predicate_bagging=False,
    random_seed=seed
)

print(f"Bags criados: {len(bags_result)}")
# Inserir classe prevista em cada bag
for bag_name, pred_dict in bags_result.items():
    for pred_rule, df_info in pred_dict.items():
        df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 180 | Descartados: 60
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bags criados: 10


In [34]:
# cov_results_dict = exp.calculate_predicate_metrics(
#     bags_result=bags_result,
#     metric='covariance',
#     threshold=0.01,
#     n_neighbors=5
# )

# # Armazenar resultados
# all_results_cov = {
#     'bags_result': bags_result,
#     'cov_results_dict': cov_results_dict
# }

# DG = exp.build_predicate_graphv2(
#     bags_result=all_results_cov['bags_result'],
#     predicate_ranking_dict=all_results_cov['cov_results_dict'],
#     metric_column='Covariance',
#     random_state=seed,
#     show_details=True
# )

# # Cálculo da Local Reaching Centrality (LRC)
# lrc_df = exp.calculate_lrc_single_graph(DG, predicates_df)
# lrc_df

pert_results_dict = exp.calculate_predicate_perturbation(
    estimator=pls_model,
    Xcalclass_prep=Xcalclass_prep,
    folds_struct=bags_result,
    predicates_df=predicates_quantiles[0],
    spectral_cuts=spectral_cuts,
    #perturbation_value=0,
    perturbation_mode='median', # valores entre 'mean' ou 'min'
    stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
    metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
    verbose=True
)

# Armazenar resultados
all_results_pert = {
    'bags_result': bags_result,
    'pert_results_dict': pert_results_dict
}

DG = exp.build_predicate_graphv2(
    bags_result=all_results_pert['bags_result'],
    predicate_ranking_dict=all_results_pert['pert_results_dict'],
    metric_column='Perturbation',
    random_state=seed,
    show_details=True
)

# Cálculo da Local Reaching Centrality (LRC)
lrc_df = exp.calculate_lrc_single_graph(DG, predicates_df)
lrc_df

PERTURBATION IMPORTANCE PARA PREDICADOS
Modo de perturbação: median
Fonte das estatísticas: full
Métrica: mean_abs_diff
Total de folds: 10


[Bag_1] Processando 182 predicados...
  Predicado: background1 > -0.07 (n=98)
    Zona: 17 colunas
    Importance: 0.000412
  Predicado: background1 <= -0.01 (n=47)
    Zona: 17 colunas
    Importance: 0.000412
  Predicado: background1 > -0.01 (n=71)
    Zona: 17 colunas
    Importance: 0.000395
  Predicado: background1 <= 0.03 (n=70)
    Zona: 17 colunas
    Importance: 0.000395
  Predicado: background1 > 0.03 (n=48)
    Zona: 17 colunas
    Importance: 0.000412
  Predicado: background1 <= 0.06 (n=94)
    Zona: 17 colunas
    Importance: 0.000402
  Predicado: Al > -0.33 (n=95)
    Zona: 15 colunas
    Importance: 0.006270
  Predicado: Al <= -0.08 (n=49)
    Zona: 15 colunas
    Importance: 0.010970
  Predicado: Al > -0.08 (n=69)
    Zona: 15 colunas
    Importance: 0.006587
  Predicado: Al <= 0.18 (n=71)
    Zona: 15 colunas
    Importance: 0.008

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ca ka > -0.38,13.636224,Ca ka,-0.38,>
1,Ca ka > -1.02,10.624870,Ca ka,-1.02,>
2,Ca ka > -1.65,9.138306,Ca ka,-1.65,>
3,Mn > -0.18,7.853342,Mn,-0.18,>
4,Si > 0.11,6.780957,Si,0.11,>
...,...,...,...,...,...
182,background3 <= 0.02,0.000597,background3,0.02,<=
183,background3 <= -0.04,0.000523,background3,-0.04,<=
184,background3 <= -0.01,0.000489,background3,-0.01,<=
185,Class_A,0.000000,None,None,None


In [35]:
lrc_df_unique = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_df_unique = lrc_df_unique.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_df_unique

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ca ka > -0.38,13.636224,Ca ka,-0.38,>
1,Mn > -0.18,7.853342,Mn,-0.18,>
2,Si > 0.11,6.780957,Si,0.11,>
3,Fe ka <= 0.82,3.147525,Fe ka,0.82,<=
4,Ti ka <= 0.87,1.402156,Ti ka,0.87,<=
5,Ca kb > -0.05,0.597448,Ca kb,-0.05,>
6,Fe kb <= -0.13,0.581948,Fe kb,-0.13,<=
7,K > 0.01,0.581869,K,0.01,>
8,Al > 0.18,0.518688,Al,0.18,>
9,P > 0.01,0.296645,P,0.01,>


## 5. Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [32]:
def extract_predicate_info(predicate_rule):
    """
    Extrai informações de uma regra de predicado.
    
    Parameters
    ----------
    predicate_rule : str
        Regra no formato "zone_name <= threshold" ou "zone_name > threshold"
    
    Returns
    -------
    dict : {'zone': str, 'operator': str, 'threshold': float}
    """
    if '<=' in predicate_rule:
        parts = predicate_rule.split('<=')
        operator = '<='
    elif '>' in predicate_rule:
        parts = predicate_rule.split('>')
        operator = '>'
    else:
        raise ValueError(f"Operador não reconhecido em: {predicate_rule}")
    
    zone_name = parts[0].strip()
    threshold_value = float(parts[1].strip())
    
    return {
        'zone': zone_name,
        'operator': operator,
        'threshold': threshold_value
    }

In [57]:
n = 0
zone_name = lrc_df.iloc[n]['Zone']
threshold_score = float(lrc_df.iloc[7]['Threshold'])  # Converter string para float
# Reconstrução: τ = mean + threshold * loadings
threshold_spectrum = reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict[zone_name]['variance_explained']:.2%}")
zone_df = spectral_zones_class[zone_name]
fig = plot_zone_with_threshold(
    zone_df=zone_df,
    threshold_spectrum=threshold_spectrum,
    zone_name=zone_name,
    class_labels=ycalclass,
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_df.iloc[n]['Node']})"
)
fig.show()


Espectro de threshold reconstruído para zona 'Ca ka':
  - Dimensão: 16 variáveis espectrais
  - Range de energias: 3.54 - 3.84
  - Variância explicada pela PC1: 99.00%


## 6. Análise de Múltiplos Thresholds

Vamos visualizar os thresholds de vários predicados importantes.

In [ ]:
def plot_zone_with_multiple_thresholds(zone_df, thresholds_dict, zone_name, 
                                        class_labels=None, title=None):
    """
    Plota uma zona espectral com múltiplos thresholds.
    
    Parameters
    ----------
    zone_df : pd.DataFrame
        DataFrame com dados espectrais da zona.
    thresholds_dict : dict
        Dicionário {label: threshold_spectrum}.
    zone_name : str
        Nome da zona espectral.
    class_labels : pd.Series, optional
        Labels de classe.
    title : str, optional
        Título do gráfico.
    """
    fig = go.Figure()
    
    x_values = pd.to_numeric(zone_df.columns, errors='coerce')
    
    # Plotar espectros das amostras (em cinza claro)
    for idx, row in zone_df.iterrows():
        fig.add_trace(go.Scatter(
            x=x_values,
            y=row.values,
            mode='lines',
            line=dict(color='rgba(128, 128, 128, 0.2)', width=0.5),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Plotar cada threshold com cor diferente
    colors = px.colors.qualitative.Set1
    for i, (label, threshold_spectrum) in enumerate(thresholds_dict.items()):
        fig.add_trace(go.Scatter(
            x=x_values,
            y=threshold_spectrum.values,
            mode='lines',
            line=dict(color=colors[i % len(colors)], width=2.5),
            name=label
        ))
    
    fig.update_layout(
        title=title or f'Zona {zone_name} - Múltiplos Thresholds',
        xaxis_title='Energia / Comprimento de Onda',
        yaxis_title='Intensidade',
        template='plotly_white'
    )
    
    return fig

In [ ]:
# ========================================
# Reconstruir thresholds para os top 5 predicados de uma zona específica
# ========================================

# Filtrar predicados da zona de interesse
zone_of_interest = zone_name  # usar a mesma zona do exemplo anterior

# Encontrar todos os predicados desta zona
zone_predicates = top_predicates[top_predicates['Zone'] == zone_of_interest]

if len(zone_predicates) > 0:
    thresholds_dict = {}
    
    for idx, row in zone_predicates.iterrows():
        pred_rule = row['Node']
        pred_info = extract_predicate_info(pred_rule)
        
        # Reconstruir threshold
        threshold_spectrum = reconstruct_threshold_to_spectrum(
            threshold_value=pred_info['threshold'],
            zone_name=pred_info['zone'],
            pca_info_dict=pca_info_dict
        )
        
        label = f"{pred_info['operator']} {pred_info['threshold']:.2f} (LRC={row['Local_Reaching_Centrality']:.3f})"
        thresholds_dict[label] = threshold_spectrum
    
    # Plotar
    fig = plot_zone_with_multiple_thresholds(
        zone_df=spectral_zones_class[zone_of_interest],
        thresholds_dict=thresholds_dict,
        zone_name=zone_of_interest,
        title=f"Zona '{zone_of_interest}' - Thresholds dos Predicados Mais Importantes"
    )
    fig.show()
else:
    print(f"Nenhum predicado encontrado para a zona '{zone_of_interest}' no top 10.")

## 7. Função Interativa para Explorar Thresholds

Função para o usuário fornecer manualmente um threshold e visualizar o espectro reconstruído.

In [ ]:
def explore_threshold(zone_name, threshold_value, spectral_zones_dict, pca_info_dict, class_labels=None):
    """
    Função interativa para explorar um threshold fornecido pelo usuário.
    
    Parameters
    ----------
    zone_name : str
        Nome da zona espectral.
    threshold_value : float
        Valor do threshold no espaço dos scores.
    spectral_zones_dict : dict
        Dicionário com dados das zonas espectrais.
    pca_info_dict : dict
        Dicionário com informações da PCA.
    class_labels : pd.Series, optional
        Labels de classe para coloração.
    
    Returns
    -------
    threshold_spectrum : pd.Series
        Espectro de threshold reconstruído.
    fig : plotly.graph_objects.Figure
        Figura com visualização.
    """
    # Validar zona
    if zone_name not in pca_info_dict:
        available_zones = list(pca_info_dict.keys())
        raise ValueError(f"Zona '{zone_name}' não encontrada. Disponíveis: {available_zones}")
    
    # Reconstruir threshold
    threshold_spectrum = reconstruct_threshold_to_spectrum(
        threshold_value=threshold_value,
        zone_name=zone_name,
        pca_info_dict=pca_info_dict
    )
    
    # Informações da PCA
    ve = pca_info_dict[zone_name]['variance_explained']
    
    print(f"\n{'='*50}")
    print(f"RECONSTRUÇÃO DE THRESHOLD")
    print(f"{'='*50}")
    print(f"Zona: {zone_name}")
    print(f"Threshold (score): {threshold_value:.4f}")
    print(f"Variância Explicada (PC1): {ve:.2%}")
    print(f"Dimensão do espectro: {len(threshold_spectrum)} variáveis")
    print(f"{'='*50}\n")
    
    # Plotar
    zone_df = spectral_zones_dict[zone_name]
    fig = plot_zone_with_threshold(
        zone_df=zone_df,
        threshold_spectrum=threshold_spectrum,
        zone_name=zone_name,
        class_labels=class_labels,
        title=f"Zona '{zone_name}' - Threshold = {threshold_value:.4f} (VE = {ve:.1%})"
    )
    
    return threshold_spectrum, fig

In [ ]:
# ========================================
# EXEMPLO: Explorar threshold manualmente
# ========================================

# Usuário pode modificar estes valores:
ZONA_ESCOLHIDA = 'F4'  # <-- MODIFICAR AQUI
THRESHOLD_ESCOLHIDO = 0.5  # <-- MODIFICAR AQUI (valor no espaço dos scores)

# Executar exploração
threshold_spec, fig = explore_threshold(
    zone_name=ZONA_ESCOLHIDA,
    threshold_value=THRESHOLD_ESCOLHIDO,
    spectral_zones_dict=spectral_zones_class,
    pca_info_dict=pca_info_dict,
    class_labels=ycalclass
)

fig.show()

In [ ]:
# ========================================
# Visualizar o espectro de threshold como tabela
# ========================================
print("Espectro de threshold reconstruído (primeiros 10 valores):")
threshold_spec.head(10)

## 8. Resumo e Comparação com Método Original

Comparação entre agregação por `extreme` e agregação por `PCA`.

In [ ]:
# ========================================
# Comparação: Agregação Extreme vs PCA
# ========================================

# Agregação original (extreme)
zone_sums_extreme = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')

# Comparar distribuições para uma zona
zone_compare = 'F4'

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=zone_sums_extreme[zone_compare],
    name='Extreme (original)',
    opacity=0.7,
    nbinsx=30
))

fig.add_trace(go.Histogram(
    x=zone_scores_df[zone_compare],
    name='PCA Scores (proposto)',
    opacity=0.7,
    nbinsx=30
))

fig.update_layout(
    title=f"Comparação de Agregações - Zona '{zone_compare}'",
    xaxis_title='Valor Agregado',
    yaxis_title='Frequência',
    barmode='overlay',
    template='plotly_white'
)

fig.show()

In [ ]:
# ========================================
# Correlação entre métodos de agregação
# ========================================
print("\nCorrelação entre Extreme e PCA Scores por zona:")
print("="*50)

for zone in zone_scores_df.columns:
    corr = np.corrcoef(zone_sums_extreme[zone], zone_scores_df[zone])[0, 1]
    ve = pca_info_dict[zone]['variance_explained']
    print(f"{zone:15s}: r = {corr:+.3f}, VE = {ve:.1%}")

## 9. Exportar Resultados

In [ ]:
# ========================================
# Salvar resultados principais
# ========================================

# DataFrame com ranking LRC
lrc_cov_df.to_csv('lrc_pca_aggregation_results.csv', index=False)
print("Resultados LRC salvos em: lrc_pca_aggregation_results.csv")

# DataFrame com informações da PCA por zona
pca_summary = pd.DataFrame({
    'Zone': list(pca_info_dict.keys()),
    'Variance_Explained': [pca_info_dict[z]['variance_explained'] for z in pca_info_dict],
    'N_Variables': [len(pca_info_dict[z]['loadings']) for z in pca_info_dict]
})
pca_summary.to_csv('pca_zones_summary.csv', index=False)
print("Resumo PCA salvo em: pca_zones_summary.csv")

pca_summary